# Spark Declarative Pipelines (SDP) — Interview Prep Guide

> A single-file reference covering every core SDP concept, from fundamentals to
> Databricks Lakeflow extensions. Written for quick revision before interviews —
> each section gives the **what**, **why**, a **code snippet**, and **interview
> talking points**.

---

## Table of Contents

1. [Core Concepts](#1-core-concepts)
2. [CLI & Tooling](#2-cli--tooling)
3. [Authoring](#3-authoring)
4. [Processing Modes](#4-processing-modes)
5. [Data Movement / Sinks](#5-data-movement--sinks)
6. [Change Data Capture (Lakeflow)](#6-change-data-capture-lakeflow)
7. [Data Quality (Lakeflow)](#7-data-quality-lakeflow)
8. [Configuration & Parameterization](#8-configuration--parameterization)
9. [Orchestration & Operations](#9-orchestration--operations)
10. [Monitoring & Governance](#10-monitoring--governance)
11. [Comparisons & Best Practices](#11-comparisons--best-practices)

---

## 1. Core Concepts

### Flows
A **flow** is the fundamental unit of data processing in SDP — a single query that
reads from a source and writes to a target dataset. Every dataset is populated by
one or more flows.

- A `CREATE STREAMING TABLE` or `@dp.table` decorated function defines a flow implicitly.
- Multiple flows can target the same dataset (see **Append Flows** below).
- Flows can be **batch** (run once per pipeline update) or **streaming** (incremental,
  checkpointed).

**Interview angle:** Be ready to explain the difference between a *dataset* and a
*flow* — a dataset is the table/view; a flow is the logic that populates it. One
dataset can have many flows (fan-in), but a flow always targets exactly one dataset.

### Datasets
SDP recognizes three dataset types:

| Type | Materialized? | Recomputation | Use Case |
|---|---|---|---|
| **Streaming Table** | Yes (physical table) | Incremental — processes only new data | Ingesting append-only/streaming sources (Kafka, Auto Loader, another streaming table) |
| **Materialized View** | Yes (physical table) | Full or incremental recompute of a query | Aggregations, joins, transformations over batch or slowly-changing data |
| **Temporary View** | No (not persisted) | Recomputed on every reference within the pipeline run | Intermediate/reusable logic scoped to a single pipeline run; not queryable outside the pipeline |

**Interview angle:** Streaming tables assume append-only sources and process data
incrementally by default; materialized views can leverage incremental refresh but
fall back to full recompute when the query isn't incrementalizable (e.g., certain
aggregations, non-deterministic functions).

### Pipelines
A **pipeline** is the top-level object that groups a set of datasets/flows together,
manages their dependency graph (DAG), handles orchestration of updates, checkpoints,
and tracks lineage. SDP infers the DAG automatically from the tables referenced in
each dataset's query (no need to manually declare a `depends_on`).

### Pipeline Projects
A **pipeline project** is the folder/file structure that defines a pipeline as code:
source files (Python/SQL), a `spark-pipeline.yml` spec file, and supporting configs.
It's the deployable unit — analogous to a repo for a traditional application.

---

## 2. CLI & Tooling

### `spark-pipelines init`
Scaffolds a new pipeline project — creates the directory structure, a sample
`spark-pipeline.yml`, and starter source files.

```bash
spark-pipelines init --name my_pipeline
```

### `spark-pipelines run`
Executes a pipeline update. Key flags:

| Flag | Behavior |
|---|---|
| *(default, no flag)* | **Incremental** — processes only new/changed data since the last run |
| `--full-refresh` | Truncates and recomputes **specific** table(s) from scratch |
| `--full-refresh-all` | Truncates and recomputes **all** tables in the pipeline from scratch |
| `--refresh` | Refresh specific table(s) incrementally (targeted subset of the pipeline) |

```bash
spark-pipelines run
spark-pipelines run --full-refresh --tables=sales_bronze
spark-pipelines run --full-refresh-all
```

**Interview angle:** Know *why* you'd need `--full-refresh` — e.g., schema changes,
corrupted checkpoints, backfilling after fixing upstream logic, or when a streaming
source's retention window has expired and state needs rebuilding.

### `spark-pipelines dry-run`
Validates the pipeline (resolves the DAG, checks syntax/dependencies) **without**
executing any data processing. Used in CI/CD to catch errors before deployment.

```bash
spark-pipelines dry-run
```

### Pipeline Spec File (`spark-pipeline.yml`)
The declarative manifest describing the pipeline: source file paths/globs, catalog
and schema targets, configuration key-value pairs, and pipeline-level settings.

```yaml
name: my_pipeline
catalog: main
schema: analytics
libraries:
  - glob:
      include: transformations/**
configuration:
  env: prod
```

---

## 3. Authoring

### Programming with SDP in Python (`pyspark.pipelines` / `dp`)
The Python API is imported conventionally as `dp` (from `pyspark.pipelines`).
Decorators define datasets declaratively; you write a function returning a
DataFrame, and SDP handles materialization, incrementalization, and scheduling.

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col

@dp.table(comment="Raw bronze ingestion")
def bronze_orders():
    return spark.readStream.table("raw.orders")

@dp.materialized_view
def gold_orders_summary():
    return spark.read.table("silver_orders").groupBy("region").count()
```

### Programming with SDP in SQL
SQL authoring uses `CREATE STREAMING TABLE` / `CREATE MATERIALIZED VIEW` DDL-style
statements inside `.sql` pipeline source files.

```sql
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM STREAM read_files('/mnt/raw/orders', format => 'json');

CREATE MATERIALIZED VIEW gold_orders_summary AS
SELECT region, COUNT(*) AS order_count
FROM silver_orders
GROUP BY region;
```

### The Spark Session in Python Pipelines
Inside SDP Python code, `spark` is **implicitly available** — you don't create a
`SparkSession` yourself (no `SparkSession.builder.getOrCreate()`). The runtime
injects a managed session scoped to the pipeline update.

**Interview angle:** A common gotcha — trying to manually construct a `SparkSession`
inside pipeline code is unnecessary and can cause conflicts; just use the
ambient `spark` object.

### Creating Materialized Views
```python
@dp.materialized_view
def daily_active_users():
    return spark.read.table("events").groupBy("date").agg(...)
```
Best for: aggregations, joins across batch tables, dimension tables, anything
needing full query semantics with automatic incremental-refresh optimization
where possible.

### Creating Temporary Views
```python
@dp.temporary_view
def cleaned_events():
    return spark.read.table("bronze_events").filter(col("is_valid"))
```
Best for: reusable intermediate logic that multiple downstream datasets reference,
without persisting a physical table.

### Creating Streaming Tables
```python
@dp.table
def bronze_events():
    return spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .load("/mnt/raw/events")
```
Best for: continuously arriving, append-only data (Kafka, Auto Loader, another
streaming table upstream).

### Loading Data from Streaming Sources
Use `spark.readStream` (Python) or `STREAM <source>` (SQL). Sources: Auto Loader
(`cloudFiles`), Kafka, Delta tables (as a stream), Kinesis, etc.

```python
@dp.table
def kafka_bronze():
    return (spark.readStream
        .format("kafka")
        .option("subscribe", "orders-topic")
        .load())
```

### Loading Data from Batch Sources
Use `spark.read` (Python) or plain `SELECT` (SQL) — for tables, files, JDBC
sources that are read wholesale on each run/refresh.

```python
@dp.materialized_view
def reference_currency_rates():
    return spark.read.format("jdbc").option(...).load()
```

### Querying Tables Defined in a Pipeline
Datasets within the same pipeline are referenced simply by name via
`spark.read.table("dataset_name")` / `spark.readStream.table("dataset_name")` —
SDP resolves the dependency and builds the DAG edge automatically; no manual
lineage wiring needed.

### Creating Tables in a For Loop (Dynamic/Parameterized Datasets)
You can generate many similarly-shaped datasets programmatically:

```python
tables = ["us", "eu", "apac"]

for region in tables:
    @dp.table(name=f"bronze_{region}")
    def _load(region=region):          # default-arg captures the loop variable
        return spark.readStream.table(f"raw.{region}_orders")
```

**Interview angle — the classic gotcha:** Python closures capture variables **by
reference**, not by value. Without `region=region` as a default argument, every
generated function would resolve `region` to the loop's *final* value at call
time (all tables would read from `apac`). This is explicitly called out as an
"Important Consideration" in SDP docs — always bind loop variables via default
arguments (or a factory function) when generating datasets dynamically.

### Using Multiple Flows to Write to a Single Target (Append Flows)
Multiple independent flows can append to the **same** streaming table target —
useful for merging several sources into one unified table without a UNION query
(which would require re-scanning everything on each batch).

```python
@dp.append_flow(target="unified_events")
def from_kafka():
    return spark.readStream.format("kafka")...

@dp.append_flow(target="unified_events")
def from_files():
    return spark.readStream.format("cloudFiles")...
```

**Interview angle:** Append flows are the mechanism for "fan-in" — many flows,
one target — and are streaming-only (target must be a streaming table).

---

## 4. Processing Modes

### Batch Processing
Runs the full query logic on each pipeline update, reading the entire source
each time. Simple, but not efficient for large/growing sources. Used for
materialized views over static/batch data.

### Incremental Processing
Only new/changed input since the last run is processed; SDP tracks progress
via checkpoints. This is the default philosophy of streaming tables, and is
applied opportunistically to materialized views when the query is
incrementalizable.

### Streaming Processing
Continuous, checkpointed consumption of unbounded/append-only sources
(Structured Streaming under the hood). Guarantees exactly-once processing
per checkpoint semantics.

### Stateless vs. Stateful Streaming Transformations
| | Stateless | Stateful |
|---|---|---|
| Definition | Each micro-batch/record processed independently | Requires tracking state across micro-batches (e.g., aggregations, joins, dedup) |
| Examples | `filter`, `select`, `map` | `groupBy().agg()`, streaming joins, `dropDuplicates` with watermark, sessionization |
| Considerations | Cheap, easily parallelized | Needs watermarking to bound state size; state stored in checkpoint |

**Interview angle:** Be ready to explain watermarking — it defines how long late
data is tolerated before state is evicted, trading correctness for bounded
memory/state size.

### Incremental Refresh (Materialized Views)
SDP attempts to refresh a materialized view incrementally (only recomputing
affected rows) rather than a full recompute, when the query pattern supports it
(e.g., simple aggregations without unsupported operators). If not supported,
it silently falls back to a full recompute — worth knowing as a
performance/cost consideration, and something to verify via the event log.

---

## 5. Data Movement / Sinks

### Writing Data to External Targets with Sinks
A **sink** lets a pipeline write output *outside* the managed catalog (e.g., to
an external Kafka topic) rather than to a Delta table inside the pipeline.

```python
dp.create_sink(
    name="kafka_sink",
    format="kafka",
    options={"kafka.bootstrap.servers": "host:9092", "topic": "out-topic"}
)

@dp.append_flow(target="kafka_sink")
def to_kafka():
    return spark.readStream.table("gold_events")
```

### Kafka Sinks
The most common sink type — streams processed data out to a Kafka topic for
downstream consumers outside the lakehouse.

### Sink Considerations
- **Streaming-only:** sinks can only be written to via streaming flows (no
  batch writes to a sink).
- **Append-only:** sinks only support append semantics — no updates/deletes/
  upserts (so CDC/AUTO CDC output can't target a sink directly).
- **Python-only:** sink definitions are currently a Python-API feature, not
  available in SQL pipeline syntax.

---

## 6. Change Data Capture (Lakeflow)

> These are Databricks Lakeflow extensions on top of open-source SDP.

### AUTO CDC (`create_auto_cdc_flow` / `AUTO CDC INTO`)
Declaratively applies a stream of change records (inserts/updates/deletes) onto
a target table, handling out-of-order arrival, keys, and sequencing
automatically — replacing hand-written `MERGE INTO` logic.

```python
dp.create_auto_cdc_flow(
    target="customers_silver",
    source="customers_cdc_bronze",
    keys=["customer_id"],
    sequence_by="updated_at",
    stored_as_scd_type=1
)
```

```sql
CREATE FLOW customers_cdc_flow
AS AUTO CDC INTO customers_silver
FROM STREAM customers_cdc_bronze
KEYS (customer_id)
SEQUENCE BY updated_at
STORED AS SCD TYPE 1;
```

### SCD Type 1
Overwrites the existing record with the latest values — **no history kept**.
Use `stored_as_scd_type=1`.

### SCD Type 2
Keeps full history — each change inserts a new row with `__START_AT`/
`__END_AT` (or similarly named) validity columns, preserving prior versions.
Use `stored_as_scd_type=2`.

### Bitemporal AUTO CDC (Beta)
Extends SCD Type 2 by tracking **two** time dimensions: the *effective* time
(when the change was true in the real world) and the *system/ingestion* time
(when it was recorded). Useful for auditing/compliance where you need to answer
"what did we believe was true, as of when."

### AUTO CDC FROM SNAPSHOT
Instead of a change-record stream, applies CDC logic by **diffing full
snapshots** taken at intervals (useful when the source system only exposes
periodic full dumps, not a true CDC/changelog feed).

```python
dp.create_auto_cdc_from_snapshot_flow(
    target="products_silver",
    source=lambda version: spark.read.table(f"snapshots.products_{version}"),
    keys=["product_id"],
    stored_as_scd_type=2
)
```

### Partial Updates / `ignore_null_updates`
When a CDC record contains `NULL` for some columns (meaning "no change to this
field," not "set to null"), `ignore_null_updates=True` preserves the target's
existing value instead of overwriting it with `NULL`.

### `apply_as_deletes` / `apply_as_truncates`
- `apply_as_deletes`: a boolean/condition expression identifying which incoming
  records represent deletes, so AUTO CDC removes the matching target row(s)
  instead of upserting them.
- `apply_as_truncates`: identifies records that signal a full truncate of the
  target table (used for full-reload-style CDC signals).

---

## 7. Data Quality (Lakeflow)

### Expectations (Concept)
Declarative data-quality constraints attached to a dataset definition. Each
expectation names a rule, defines a boolean SQL condition, and specifies what
happens to records that violate it.

### Expectation Name
A human-readable identifier for the rule (shows up in metrics/dashboards),
e.g. `"valid_order_amount"`.

### Constraint to Evaluate
The boolean SQL expression checked per row, e.g. `"order_amount > 0"`.

### Action on Invalid Record
| Action | Behavior |
|---|---|
| **warn** (default via `expect`) | Row is kept; violation is counted/logged |
| **drop** | Row is filtered out of the output |
| **fail** | Entire pipeline update fails immediately on violation |

### `expect`, `expect_or_drop`, `expect_or_fail`
```python
@dp.table
@dp.expect("valid_amount", "order_amount > 0")                 # warn
@dp.expect_or_drop("valid_email", "email IS NOT NULL")         # drop
@dp.expect_or_fail("valid_currency", "currency IN ('USD','EUR')")  # fail
def orders_silver():
    return spark.readStream.table("orders_bronze")
```

```sql
CREATE STREAMING TABLE orders_silver (
  CONSTRAINT valid_amount EXPECT (order_amount > 0),
  CONSTRAINT valid_email EXPECT (email IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_currency EXPECT (currency IN ('USD','EUR')) ON VIOLATION FAIL UPDATE
)
AS SELECT * FROM STREAM orders_bronze;
```

### `expect_all`, `expect_all_or_drop`, `expect_all_or_fail`
Same three actions, but applied to a **dictionary of multiple constraints** at
once, more concise than stacking many single decorators:

```python
rules = {
    "valid_amount": "order_amount > 0",
    "valid_email": "email IS NOT NULL"
}

@dp.table
@dp.expect_all_or_drop(rules)
def orders_silver():
    return spark.readStream.table("orders_bronze")
```

### Expectation Tracking Metrics
SDP automatically records pass/fail counts per expectation in the **pipeline
event log**, enabling data-quality dashboards without custom instrumentation.

### Quarantining Invalid Records
A common pattern (not a single built-in keyword, but standard practice):
route failing records to a separate "quarantine" table instead of just
dropping them, typically by writing two versions of the table — one with
`expect_or_drop` for the clean path, and a second flow capturing the inverse
condition (`NOT (constraint)`) into a quarantine table for later inspection/
reprocessing.

---

## 8. Configuration & Parameterization

### Pipeline Parameters (SQL, `${param}` syntax)
SQL pipeline code can reference parameters defined in the pipeline's
configuration using `${param_name}` substitution:

```sql
CREATE STREAMING TABLE bronze_orders
AS SELECT * FROM STREAM read_files('${source_path}');
```

### Configuration Field (Python, `spark.conf.get()`)
In Python, the equivalent is reading pipeline configuration via Spark conf:

```python
source_path = spark.conf.get("source_path")

@dp.table
def bronze_orders():
    return spark.readStream.format("cloudFiles").load(source_path)
```

### Parameter Precedence (job run > job > task > pipeline)
When the same configuration key is set at multiple levels, the **most specific,
most immediate override wins**:

```
job run  >  job  >  task  >  pipeline
(highest precedence)          (lowest precedence)
```

**Interview angle:** This mirrors typical override hierarchies (like CSS
specificity or env-var layering) — a one-off manual job run parameter beats
everything, while the pipeline's own default config is the fallback baseline.

---

## 9. Orchestration & Operations

### Pipeline Orchestrating in a Job (Lakeflow Jobs / Pipeline Task)
Pipelines are typically triggered as a **task** inside a Databricks Job
(Lakeflow Jobs), allowing them to be chained with other tasks (notebooks,
SQL, dbt), scheduled, and monitored alongside the rest of a workflow.

### Orchestrating with Apache Airflow (`DatabricksSubmitRunOperator`)
External orchestration from Airflow uses Databricks provider operators to
trigger a pipeline update as part of a broader DAG:

```python
from airflow.providers.databricks.operators.databricks import DatabricksSubmitRunOperator

run_pipeline = DatabricksSubmitRunOperator(
    task_id="run_sdp_pipeline",
    databricks_conn_id="databricks_default",
    json={"pipeline_task": {"pipeline_id": "<pipeline-id>"}}
)
```

### Orchestrating with Azure Data Factory (REST API)
ADF has no native SDP connector — pipelines are triggered via a **Web
Activity** calling the Databricks Jobs/Pipelines REST API directly (e.g.,
`POST /api/2.0/pipelines/{pipeline_id}/updates`).

### Continuous vs. Triggered Pipeline Mode
| Mode | Behavior |
|---|---|
| **Triggered** | Runs once, processes available data, then stops (good for scheduled batch-like cadence) |
| **Continuous** | Stays running, processing new data as it arrives with minimal latency (true streaming) |

### Retry Behavior / `pipelines.numUpdateRetryAttempts`
Controls how many times SDP automatically retries a failed pipeline update
before surfacing the failure — configurable via the
`pipelines.numUpdateRetryAttempts` setting, useful for transient
infrastructure/source issues.

---

## 10. Monitoring & Governance

### Pipeline Event Log
A structured Delta table automatically maintained by SDP capturing every
pipeline update's lifecycle: flow progress, data quality metric events, errors,
lineage details. It's queryable like any table — the backbone for building
custom monitoring.

```sql
SELECT * FROM event_log(TABLE(my_pipeline)) WHERE event_type = 'flow_progress';
```

### Data Quality Metrics / Dashboards
Expectation pass/fail counts (from the event log) can be surfaced in
dashboards (e.g., Databricks SQL dashboards) to track data quality trends
over time per dataset/expectation.

### Event Hooks (Custom Monitoring)
Mechanism to attach custom callback logic that fires on pipeline events
(e.g., posting to Slack/PagerDuty on failure), integrating SDP into external
alerting systems.

### Unity Catalog Integration
SDP pipelines register their output datasets directly into **Unity Catalog**,
inheriting UC's governance features: fine-grained access control, lineage
tracking (automatically links source → transformation → output across the
whole pipeline DAG), and auditing — all without extra configuration.

---

## 11. Comparisons & Best Practices

### SDP vs. Traditional/Procedural Spark Programming
| | Traditional Spark | SDP |
|---|---|---|
| Style | Imperative — you write the *how* (read, transform, write, schedule, checkpoint management) | Declarative — you describe the *what* (desired table = this query); the engine figures out execution |
| Dependency management | Manual (you sequence jobs/notebooks yourself) | Automatic DAG inference from table references |
| Checkpointing/retries | Manual setup | Built-in |
| Data quality | Custom code | Built-in expectations |

### Procedural vs. Declarative Data Processing
The core paradigm shift: procedural code specifies *execution steps*;
declarative code specifies *desired end state*, leaving the engine to
determine the optimal execution plan, incremental strategy, and retry/
recovery behavior. This is the same paradigm shift as SQL vs. hand-rolled
loops over rows.

### SDP vs. Databricks Lakeflow Pipelines (Feature Comparison)
**Spark Declarative Pipelines (SDP)** is the **open-source** engine (part of
Apache Spark) providing the core declarative pipeline framework — flows,
datasets, the DAG engine, expectations basics. **Databricks Lakeflow
Pipelines** is Databricks' managed product built on top of SDP, adding
proprietary/platform extensions:

| Feature | SDP (OSS) | Lakeflow Pipelines (Databricks) |
|---|---|---|
| Flows, datasets, DAG inference | ✅ | ✅ |
| Basic expectations | ✅ | ✅ |
| AUTO CDC / SCD Type 1 & 2 / Bitemporal | ❌ | ✅ |
| Sinks (Kafka, external) | Partial/evolving | ✅ |
| Managed serverless compute, auto-scaling | ❌ | ✅ |
| Unity Catalog governance integration | ❌ (OSS has no UC) | ✅ |
| Event log dashboards, UI | ❌ | ✅ |

**Interview angle:** This is a favorite "do you actually understand the
ecosystem" question — know that Lakeflow = SDP + Databricks-managed
infrastructure + proprietary CDC/governance/monitoring extensions.

### Medallion Architecture (Bronze/Silver/Gold) with SDP
SDP maps naturally onto the medallion pattern:

- **Bronze:** raw ingestion, typically streaming tables reading directly from
  source (files/Kafka), minimal transformation.
- **Silver:** cleaned, conformed, deduplicated data — often where expectations
  (`expect_or_drop`) and AUTO CDC merges are applied.
- **Gold:** business-level aggregates and marts — typically materialized
  views, consumed by BI tools/ML.

The pipeline's auto-inferred DAG naturally reflects bronze → silver → gold
lineage since each layer's query references the prior layer's table by name.

### Important Considerations (Python & SQL)
**Forbidden/unsupported operations** inside pipeline query definitions — these
break incrementalization or aren't supported inside the managed dataflow
graph:
- `collect()` — pulls data to the driver, breaks distributed/incremental
  execution model.
- `count()` (as a standalone action inside pipeline logic) — triggers eager
  execution outside the managed flow.
- `pivot()` — not supported in the incremental/streaming dataflow engine.
- `PIVOT` (SQL) — same restriction, called out explicitly for SQL pipelines.
- General rule: avoid **actions** (eager triggers) inside dataset-defining
  functions — only return a lazy DataFrame/query; let SDP manage execution.

**For-loop closure gotchas:** (detailed above under *Creating Tables in a For
Loop*) — always bind loop variables via default arguments when
dynamically generating dataset functions, or every generated table silently
reads from the same (last) loop value.

---

## Quick-Fire Interview Recap

- **Flow vs Dataset:** flow = the query/logic; dataset = the table/view it populates.
- **3 dataset types:** streaming table (incremental, append-only sources), materialized view (batch/aggregation, opportunistic incremental refresh), temporary view (not persisted, pipeline-scoped).
- **Pipeline vs Pipeline Project:** pipeline = the running/managed DAG object; project = the source code + spec file that defines it.
- **`--full-refresh` vs `--full-refresh-all`:** targeted table(s) vs. entire pipeline.
- **AUTO CDC replaces:** hand-written `MERGE INTO` upsert logic, with built-in SCD1/SCD2/bitemporal support.
- **Expectation actions:** warn (default) → drop → fail, in increasing strictness.
- **Parameter precedence:** job run > job > task > pipeline.
- **Sinks are:** streaming-only, append-only, Python-only.
- **SDP vs Lakeflow Pipelines:** SDP = OSS core engine; Lakeflow = Databricks managed superset (CDC, UC governance, dashboards, serverless compute).
- **Forbidden calls:** `collect()`, `count()`, `pivot()` / `PIVOT` — breaks the declarative/incremental execution model.